# 🎮 Colab Gaming PC (L4 GPU / High-Memory) - クラウドゲーミング＆サーバー統合環境

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/9z2fqjckjv-bot/ColabTool/blob/main/GamingPC/GamingPC_Colab.ipynb)

Google Colabの強力な **NVIDIA L4 GPU (24GB VRAM) & ハイメモリ (53GB RAM)** を活用し、本格的なクラウドゲーミングワークステーションおよび自立型Minecraftサーバーをワンクリックで構築・稼働させる自動化ノートブックです。

---

### 🌟 本環境の主な機能と仕様
- **🖥️ Xfce4 デスクトップ & 仮想オーディオ**: 1920x1080 (Full HD) の高解像度デスクトップと、PulseAudio 仮想サウンドデバイスを完備。
- **🌐 デュアル・リモートアクセス方式**:
  - **noVNC (Webブラウザ直接アクセス)**: 特別なソフトのインストール不要。自動生成されるHTTPSリンクをクリックするだけで、ブラウザ内で即座にデスクトップを操作可能。
  - **Chrome Remote Desktop (CRD)**: ゲームプレイに最適な高フレームレート・低遅延・全画面表示対応。
- **🌐 Google Chrome マルチプロファイル**: Profile 1, 2, 3 の個別ショートカットをデスクトップに配置。タブの展開やアカウントの使い分けが快適に行えます。
- **🎮 充実のゲーミング・ランチャー環境**:
  - **Steam**: Linux / Proton互換環境により、Hearts of Iron IV (Hoi4) などのSteamゲームをバックグラウンド実行・プレイ可能。
  - **Minecraft (Prism Launcher)**: Modrinth/CurseForgeのMODやリソースパックをワンクリックで導入できる高性能Minecraftクライアント。
  - **CurseForge**: 各種ゲームMODの一括管理。
  - **Xbox Cloud Gaming**: 専用Chrome WebAppショートカットで即座にクラウドゲームを起動。
  - **Roblox**: デスクトップショートカットから直接アクセス。
  - **ChillwithYou**: リラックス・作業用BGMコンパニオンをバックグラウンド稼働。
- **⛏️ セル非依存・独立稼働型 Minecraft サーバー**: PaperMC 1.21.1 (最新安定版) をバックグラウンドデーモンとして起動。playit.ggトンネルにより外部からのマルチプレイ接続にも対応。
- **🎥 L4 GPU ハードウェアアクセラレーション画面収録 (NVENC 60fps)**: NVIDIA L4のNVENCエンジンを活用し、CPUやゲーム描画に負荷をかけずに1080p 60fpsの高画質デスクトップ＆音声収録が可能（OBS Studioも併せて完備）。
- **💬 Discord**: デスクトップアプリ＆Web版を完備。
- **💾 Google Drive 完全データ永続化**: ワールドデータ、Steamセーブ、ブラウザプロファイル、MOD構成、録画動画を `/content/drive/MyDrive/GamingPC/` に自動同期。次回起動時にシームレスに復元されます。
- **⚡ Colab非同期・独立稼働**: セル実行は約1〜2分で自動完了して終了するため、後続のセルやColabの通常作業をブロックしません。

---
## 🚀 ステップ 1: 【ワンクリック起動】Gaming PC & 独立Minecraftサーバー

下のセルを実行（再生ボタンを押す）するだけで、すべての環境構築・デスクトップ起動・Minecraftサーバー立ち上げが全自動で始まります。  
バックグラウンドで独立稼働するため、**約1〜2分でこのセルの実行は正常完了**します。

> [!TIP]
> **接続方法の選択**:
> - **noVNC (おすすめ・設定不要)**: セル完了時に表示される `trycloudflare.com` のリンクをクリックするだけで、ブラウザ内で即座にデスクトップが開きます。
> - **Chrome Remote Desktop (最高画質)**: [Google Remote Desktop Headless](https://remotedesktop.google.com/headless) で認証コードを取得し、下の `CRD_AUTH_CODE` に貼り付けて実行してください（起動後に追加設定することも可能です）。

In [ ]:
#@title 🎮 ステップ 1: 【ワンクリック起動】Gaming PC & 独立Minecraftサーバー
import os
import sys
import json
import subprocess
import time
from IPython.display import display, HTML

#@markdown ### 🌐 リモート接続方式の選択
REMOTE_METHOD = "\u4e21\u65b9\u8d77\u52d5 (noVNC + Chrome Remote Desktop \u63a8\u5968)" #@param ["noVNC (Webブラウザで即座にアクセス・設定不要)", "Chrome Remote Desktop (最高画質・低遅延・ゲーム推奨)", "両方起動 (noVNC + Chrome Remote Desktop 推奨)"]
CRD_AUTH_CODE = "" #@param {type:"string"}
CRD_PIN = "123456" #@param {type:"string"}

#@markdown ### ⛏️ Minecraftサーバー設定
START_MINECRAFT_SERVER = True #@param {type:"boolean"}
MC_RAM_GB = 8 #@param {type:"slider", min:4, max:16, step:2}

#@markdown ### 💾 Google Drive バックアップ
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}

print("==================================================")
print("  🎮 Colab Gaming PC & 独立Minecraftサーバー 起動")
print("==================================================")

# 1. Google Drive の自動マウント
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("📂 Google Drive をマウント中...")
            drive.mount("/content/drive")
            print("✅ Google Drive マウント完了")
        else:
            print("📂 Google Drive は既にマウントされています。")
    except Exception as e:
        print("⚠️ Google Driveマウントをスキップ (ローカル保存で継続):", e)

# 2. スクリプト群の自己展開
REPO_DIR = "/content/ColabTool/GamingPC"
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")
os.makedirs(SCRIPTS_DIR, exist_ok=True)

EMBEDDED_FILES = {"scripts/setup_environment.sh": "#!/bin/bash\n# ==============================================================================\n# setup_environment.sh - Gaming PC on Google Colab (L4 GPU High-Memory) Setup\n# ==============================================================================\nset -e\n\nexport DEBIAN_FRONTEND=noninteractive\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\n\necho \"==================================================\"\necho \"  🎮 Gaming PC 仮想環境 自動セットアップ開始\"\necho \"  動作環境: Colab L4 GPU (24GB VRAM) / High-RAM\"\necho \"==================================================\"\n\n# 1. 32-bit アーキテクチャの有効化（Steam必須）\necho \">> [1/8] 32bitアーキテクチャの追加 & パッケージ更新...\"\ndpkg --add-architecture i386 || true\napt-get update -qq\n\n# 2. Xfce4 デスクトップ環境, 音声 (PulseAudio), OpenGL/Vulkan, VNC, noVNC\necho \">> [2/8] Xfce4 デスクトップ・音声・OpenGL/Vulkan・画面描画ツールのインストール...\"\napt-get install -y -qq \\\n    xfce4 xfce4-goodies dbus-x11 x11-xserver-utils x11-utils xterm \\\n    xvfb x11vnc websockify novnc \\\n    pulseaudio pulseaudio-utils alsa-utils pavucontrol \\\n    libgl1-mesa-glx mesa-utils libvulkan1 vulkan-tools \\\n    ffmpeg jq curl wget git zstd htop \\\n    python3-dbus python3-gi libdbus-1-3 libdbus-1-dev libglib2.0-dev python3-dev pkg-config \\\n    openjdk-21-jdk-headless > /dev/null 2>&1\n\n# Python D-Bus バインディングの確実なコンパイル・導入 (Colab Python用)\npython3 -m pip install --ignore-installed --no-cache-dir dbus-python > /dev/null 2>&1 || true\nif which python3.10 >/dev/null 2>&1 && python3.10 -c \"import dbus\" >/dev/null 2>&1; then\n    sed -i '1s|.*|#!/usr/bin/python3.10|' /opt/google/chrome-remote-desktop/chrome-remote-desktop 2>/dev/null || true\nfi\n\n# 3. Google Chrome & Chrome Remote Desktop のインストール\necho \">> [3/8] Google Chrome & Chrome Remote Desktop の確認 & インストール...\"\nif ! which google-chrome > /dev/null 2>&1 && ! which google-chrome-stable > /dev/null 2>&1; then\n    wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb\n    apt-get install -y -qq /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1\n    rm -f /tmp/chrome.deb\nfi\n\nif ! dpkg -s chrome-remote-desktop > /dev/null 2>&1; then\n    wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb\n    apt-get install -y -qq /tmp/crd.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1\n    rm -f /tmp/crd.deb\nfi\n\n# CRD用一般ユーザー (colabuser) の作成 (CRDはrootでの直接実行を拒否するため)\nif ! id -u colabuser >/dev/null 2>&1; then\n    useradd -m -s /bin/bash -G sudo,chrome-remote-desktop colabuser\n    echo \"colabuser:123456\" | chpasswd\n    echo \"colabuser ALL=(ALL) NOPASSWD:ALL\" > /etc/sudoers.d/colabuser\nfi\n\nmkdir -p /root/.config/chrome-remote-desktop /home/colabuser/.config/chrome-remote-desktop\necho \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /root/.chrome-remote-desktop-session\necho \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /home/colabuser/.chrome-remote-desktop-session\nchmod +x /root/.chrome-remote-desktop-session /home/colabuser/.chrome-remote-desktop-session\nchown -R colabuser:colabuser /home/colabuser\n\n# 4. Steam のインストール\necho \">> [4/8] Steam の確認 & インストール...\"\nif ! which steam > /dev/null 2>&1; then\n    apt-get install -y -qq steam-installer > /dev/null 2>&1 || apt-get install -y -qq steam > /dev/null 2>&1 || true\nfi\n\n# 5. Discord のインストール\necho \">> [5/8] Discord の確認 & インストール...\"\nif ! which discord > /dev/null 2>&1; then\n    wget -q -O /tmp/discord.deb \"https://discord.com/api/download?platform=linux&format=deb\" || true\n    if [ -f /tmp/discord.deb ]; then\n        apt-get install -y -qq /tmp/discord.deb > /dev/null 2>&1 || apt-get install -f -y -qq > /dev/null 2>&1 || true\n        rm -f /tmp/discord.deb\n    fi\nfi\n\n# 6. CurseForge & Prism Launcher (Minecraftクライアント) & OBS Studio\necho \">> [6/8] CurseForge / Prism Launcher / OBS Studio のセットアップ...\"\nmkdir -p /usr/local/bin /content/tools\n\n# CurseForge AppImage\nif [ ! -f /usr/local/bin/curseforge ]; then\n    curl -sSL -o /usr/local/bin/curseforge \"https://curseforge.overwolf.com/downloads/curseforge-latest-linux.AppImage\" || true\n    chmod +x /usr/local/bin/curseforge 2>/dev/null || true\nfi\n\n# Prism Launcher (高性能Minecraftランチャー)\nif [ ! -f /usr/local/bin/prismlauncher ]; then\n    curl -sSL -o /usr/local/bin/prismlauncher \"https://github.com/PrismLauncher/PrismLauncher/releases/download/8.4/PrismLauncher-Linux-8.4-x86_64.AppImage\" || true\n    chmod +x /usr/local/bin/prismlauncher 2>/dev/null || true\nfi\n\n# OBS Studio\nif ! which obs > /dev/null 2>&1; then\n    apt-get install -y -qq obs-studio > /dev/null 2>&1 || true\nfi\n\n# 7. トンネルツール (Cloudflared & playit.gg)\necho \">> [7/8] トンネルツール (Cloudflared & playit.gg) のセットアップ...\"\nif [ ! -f /usr/local/bin/cloudflared ]; then\n    curl -sSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\n    chmod +x /usr/local/bin/cloudflared\nfi\n\nif [ ! -f /usr/local/bin/playit ]; then\n    curl -sSL -o /usr/local/bin/playit https://github.com/playit-cloud/playit-agent/releases/download/v0.15.26/playit-linux-amd64\n    chmod +x /usr/local/bin/playit\nfi\n\n# 8. デスクトップ環境・ショートカット・プロファイル整備\necho \">> [8/8] デスクトップショートカット & プロファイル設定...\"\nDESKTOP_DIR=\"/root/Desktop\"\nmkdir -p \"$DESKTOP_DIR\" /content/chrome_profiles/profile1 /content/chrome_profiles/profile2 /content/chrome_profiles/profile3 /content/recordings\n\n# Chrome Profile 1 ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Chrome_Profile1.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Chrome (Profile 1)\nComment=Google Chrome Main Profile\nExec=google-chrome --no-sandbox --user-data-dir=/content/chrome_profiles/profile1\nIcon=google-chrome\nTerminal=false\nCategories=Network;WebBrowser;\nEOF\n\n# Chrome Profile 2 ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Chrome_Profile2.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Chrome (Profile 2)\nComment=Google Chrome Sub Profile\nExec=google-chrome --no-sandbox --user-data-dir=/content/chrome_profiles/profile2\nIcon=google-chrome\nTerminal=false\nCategories=Network;WebBrowser;\nEOF\n\n# Steam ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Steam.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Steam\nComment=Steam Gaming Platform\nExec=steam\nIcon=steam\nTerminal=false\nCategories=Game;\nEOF\n\n# Discord ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Discord.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Discord\nComment=Discord Voice & Chat\nExec=discord --no-sandbox\nIcon=discord\nTerminal=false\nCategories=Network;InstantMessaging;\nEOF\n\n# Minecraft Launcher (Prism Launcher) ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Minecraft_Prism.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Minecraft (Prism Launcher)\nComment=Prism Launcher for Minecraft & Modpacks\nExec=/usr/local/bin/prismlauncher --appimage-extract-and-run\nIcon=system-run\nTerminal=false\nCategories=Game;\nEOF\n\n# CurseForge ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/CurseForge.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=CurseForge\nComment=CurseForge Mod Manager\nExec=/usr/local/bin/curseforge --no-sandbox --appimage-extract-and-run\nIcon=applications-games\nTerminal=false\nCategories=Game;\nEOF\n\n# Xbox Cloud Gaming ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Xbox_Cloud_Gaming.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Xbox Cloud Gaming\nComment=Xbox Cloud Gaming on Browser\nExec=google-chrome --no-sandbox --user-data-dir=/content/chrome_profiles/profile1 --app=https://www.xbox.com/play\nIcon=input-gaming\nTerminal=false\nCategories=Game;\nEOF\n\n# Roblox Web / Sober ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Roblox.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=Roblox\nComment=Roblox Gaming Platform\nExec=google-chrome --no-sandbox --user-data-dir=/content/chrome_profiles/profile1 --app=https://www.roblox.com\nIcon=applications-games\nTerminal=false\nCategories=Game;\nEOF\n\n# ChillwithYou ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/ChillwithYou.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=ChillwithYou\nComment=Chill with You / Lo-Fi Audio & Relax Companion\nExec=google-chrome --no-sandbox --user-data-dir=/content/chrome_profiles/profile1 --app=https://chillhop.com/live/chillhop-radio/\nIcon=audio-headphones\nTerminal=false\nCategories=AudioVideo;Audio;\nEOF\n\n# 1-Click 画面収録 (NVENC 60fps) ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/ScreenRecorder_Toggle.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=🎥 画面収録 (開始/停止)\nComment=Toggle Hardware Accelerated Screen Recording\nExec=/bin/bash -c \"/content/ColabTool/GamingPC/scripts/record_screen.sh toggle\"\nIcon=media-record\nTerminal=true\nCategories=AudioVideo;Recorder;\nEOF\n\n# OBS Studio ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/OBS_Studio.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=OBS Studio\nComment=OBS Studio Screen & Stream Recording\nExec=obs\nIcon=obs\nTerminal=false\nCategories=AudioVideo;Recorder;\nEOF\n\n# Minecraft サーバー ログ監視コンソール\ncat << 'EOF' > \"$DESKTOP_DIR/Minecraft_Server_Logs.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=⛏️ Minecraft サーバー ログ監視\nComment=View background Minecraft server logs\nExec=xterm -title \"Minecraft Server Logs\" -hold -e \"tail -n 100 -f /content/minecraft_server/logs/latest.log\"\nIcon=utilities-terminal\nTerminal=false\nCategories=System;\nEOF\n\n# 音声ミキサー (pavucontrol) ショートカット\ncat << 'EOF' > \"$DESKTOP_DIR/Volume_Control.desktop\"\n[Desktop Entry]\nVersion=1.0\nType=Application\nName=🔊 音量・サウンド設定\nComment=PulseAudio Volume Control\nExec=pavucontrol\nIcon=audio-volume-high\nTerminal=false\nCategories=AudioVideo;\nEOF\n\n# 実行権限の付与 & colabuserへの同期\nchmod +x \"$DESKTOP_DIR\"/*.desktop 2>/dev/null || true\nmkdir -p /home/colabuser/Desktop\ncp -rf \"$DESKTOP_DIR\"/* /home/colabuser/Desktop/ 2>/dev/null || true\nchmod +x /home/colabuser/Desktop/*.desktop 2>/dev/null || true\nchown -R colabuser:colabuser /home/colabuser\n\necho \"✅ [完了] Gaming PC 仮想環境のセットアップが正常に完了しました。\"\n", "scripts/start_desktop.sh": "#!/bin/bash\n# ==============================================================================\n# start_desktop.sh - Start Xfce4, Virtual Audio, VNC, noVNC & Tunnel\n# ==============================================================================\nset -e\n\nCRD_CODE=\"$1\"\nCRD_PIN=\"${2:-123456}\"\n\necho \"==================================================\"\necho \"  🖥️ デスクトップ環境 & リモート接続の起動\"\necho \"==================================================\"\n\n# 1. 仮想オーディオ (PulseAudio) の起動\necho \">> [1/4] 仮想オーディオ (PulseAudio) を初期化中...\"\npulseaudio -D --exit-idle-time=-1 2>/dev/null || true\npactl load-module module-null-sink sink_name=VirtualSink sink_properties=device.description=VirtualSink 2>/dev/null || true\npactl set-default-sink VirtualSink 2>/dev/null || true\n\n# 2. Xvfb 仮想ディスプレイ (:1, 1920x1080 24bit) の起動\necho \">> [2/4] Xvfb 仮想ディスプレイ (:1, 1080p) & Xfce4 を起動中...\"\nexport DISPLAY=:1\nif ! pgrep -f \"Xvfb :1\" > /dev/null 2>&1; then\n    nohup Xvfb :1 -screen 0 1920x1080x24 +extension GLX +extension RENDER -noreset > /content/xvfb.log 2>&1 &\n    sleep 2\nfi\n\nif ! pgrep -f \"xfce4-session\" > /dev/null 2>&1; then\n    nohup dbus-launch --exit-with-session startxfce4 > /content/xfce.log 2>&1 &\n    sleep 2\nfi\n\n# 3. x11vnc & noVNC (Webブラウザ用リモートデスクトップ) の起動\necho \">> [3/4] x11vnc & noVNC サーバーをバックグラウンド起動中...\"\npkill -f \"x11vnc.*:1\" 2>/dev/null || true\nnohup x11vnc -display :1 -nopw -listen 0.0.0.0 -xkb -ncache 10 -ncache_cr -forever -shared -rfbport 5901 > /content/x11vnc.log 2>&1 &\n\npkill -f \"websockify.*6080\" 2>/dev/null || true\nNOVNC_DIR=\"/usr/share/novnc\"\nif [ ! -d \"$NOVNC_DIR\" ]; then\n    NOVNC_DIR=\"/usr/local/share/novnc\"\nfi\nnohup websockify --web=\"$NOVNC_DIR\" 6080 localhost:5901 > /content/novnc.log 2>&1 &\nsleep 2\n\n# Cloudflared 高速トンネルの起動 (noVNC を HTTPS で直接アクセス可能にする)\npkill -f \"cloudflared tunnel\" 2>/dev/null || true\nrm -f /content/cloudflared.log\nnohup cloudflared tunnel --url http://localhost:6080 > /content/cloudflared.log 2>&1 &\n\necho \"Cloudflare トンネルURLの取得待機中...\"\nNOVNC_URL=\"\"\nfor i in {1..20}; do\n    NOVNC_URL=$(grep -oE \"https://[a-zA-Z0-9.-]+\\.trycloudflare\\.com\" /content/cloudflared.log 2>/dev/null | head -n 1 || true)\n    if [ -n \"$NOVNC_URL\" ]; then\n        break\n    fi\n    sleep 1\ndone\n\n# 4. Chrome Remote Desktop (CRD) の設定（コードが指定されている場合）\necho \">> [4/4] リモート接続方式の構成...\"\nif [ -n \"$CRD_CODE\" ]; then\n    echo \"Chrome Remote Desktop を構成中...\"\n    if ! python3 -c \"import dbus\" > /dev/null 2>&1; then\n        echo \"python3-dbus パッケージをインストール & ビルド中...\"\n        apt-get install -y -qq python3-dbus python3-gi libdbus-1-3 libdbus-1-dev libglib2.0-dev python3-dev pkg-config > /dev/null 2>&1 || true\n        python3 -m pip install --ignore-installed --no-cache-dir dbus-python > /dev/null 2>&1 || true\n        if which python3.10 >/dev/null 2>&1 && python3.10 -c \"import dbus\" >/dev/null 2>&1; then\n            sed -i '1s|.*|#!/usr/bin/python3.10|' /opt/google/chrome-remote-desktop/chrome-remote-desktop 2>/dev/null || true\n        fi\n    fi\n\n    # colabuserの存在確認・設定 (CRDはroot実行不可のため)\n    if ! id -u colabuser >/dev/null 2>&1; then\n        useradd -m -s /bin/bash -G sudo,chrome-remote-desktop colabuser\n        echo \"colabuser:123456\" | chpasswd\n        echo \"colabuser ALL=(ALL) NOPASSWD:ALL\" > /etc/sudoers.d/colabuser\n    fi\n    mkdir -p /home/colabuser/.config/chrome-remote-desktop\n    echo \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /home/colabuser/.chrome-remote-desktop-session\n    echo \"exec /etc/X11/Xsession /usr/bin/xfce4-session\" > /etc/chrome-remote-desktop-session\n    chmod +x /home/colabuser/.chrome-remote-desktop-session /etc/chrome-remote-desktop-session\n    adduser colabuser chrome-remote-desktop 2>/dev/null || true\n    chown -R colabuser:colabuser /home/colabuser\n\n    # コマンド判定（全文コピーか、認証コード単体か）\n    if [[ \"$CRD_CODE\" == *\"start-host\"* ]]; then\n        CMD=\"$CRD_CODE\"\n        if [[ \"$CMD\" != *\"--pin=\"* ]]; then\n            CMD=\"$CMD --pin=$CRD_PIN\"\n        fi\n    else\n        CMD=\"DISPLAY= /opt/google/chrome-remote-desktop/start-host --code=\\\"$CRD_CODE\\\" --redirect-url=\\\"https://remotedesktop.google.com/_/oauthredirect\\\" --name=\\$(hostname) --pin=$CRD_PIN\"\n    fi\n\n    echo \"CRD ホスト起動を実行中 (ユーザー: colabuser)...\"\n    service dbus start > /dev/null 2>&1 || true\n    sudo -u colabuser bash -c \"$CMD\" > /content/crd_setup.log 2>&1 || true\n    nohup sudo -u colabuser /opt/google/chrome-remote-desktop/chrome-remote-desktop --foreground > /content/crd_run.log 2>&1 &\n    echo \"✅ Chrome Remote Desktop を構成・起動しました (ログ: /content/crd_run.log)\"\n    sleep 2\n    tail -n 10 /content/crd_run.log 2>/dev/null || true\nfi\n\necho \"\"\necho \"==================================================\"\necho \"🎉 デスクトップ環境の起動が完了しました！\"\necho \"==================================================\"\nif [ -n \"$NOVNC_URL\" ]; then\n    echo \"🌐 【ブラウザ即座アクセス用 URL (noVNC)】:\"\n    echo \"👉  ${NOVNC_URL}/vnc.html?autoconnect=true&resize=scale\"\n    echo \"--------------------------------------------------\"\nfi\nif [ -n \"$CRD_CODE\" ]; then\n    echo \"🖥️ 【Chrome Remote Desktop】:\"\n    echo \"👉  https://remotedesktop.google.com/access\"\n    echo \"    (設定PIN: $CRD_PIN)\"\n    echo \"--------------------------------------------------\"\nfi\necho \"✨ 画面解像度: 1920x1080 (Full HD)\"\necho \"✨ デスクトップ上にChrome(各プロファイル), Steam, Discord, Minecraft, CurseForge等のショートカットが配置されています。\"\necho \"==================================================\"\n", "scripts/start_mc_server.sh": "#!/bin/bash\n# ==============================================================================\n# start_mc_server.sh - Decoupled Background Minecraft Paper Server\n# (Runs independently of notebook cells; uses named pipe for commands)\n# ==============================================================================\nset -e\n\nMC_RAM_GB=\"${1:-8}\"\nSERVER_DIR=\"/content/minecraft_server\"\nMINECRAFT_VERSION=\"1.21.1\"\n\necho \"==================================================\"\necho \"  ⛏️ 独立Minecraftサーバー (PaperMC) 起動\"\necho \"  割り当てRAM: ${MC_RAM_GB}GB (Colab High-RAM最適化)\"\necho \"==================================================\"\n\nmkdir -p \"$SERVER_DIR/logs\" \"$SERVER_DIR/plugins\"\ncd \"$SERVER_DIR\"\n\n# EULA 同意\necho \"eula=true\" > eula.txt\n\n# server.properties 最適化\nif [ ! -f \"server.properties\" ]; then\ncat << 'EOF' > server.properties\nserver-port=25565\nonline-mode=false\ndifficulty=normal\ngamemode=survival\npvp=true\nview-distance=10\nsimulation-distance=8\nmax-players=20\nenable-command-block=true\nmotd=Colab Gaming PC Minecraft Server\nnetwork-compression-threshold=256\nsync-chunk-writes=false\nallow-flight=true\nspawn-protection=0\nEOF\nfi\n\n# PaperMC jar のダウンロード (未ダウンロード時)\nif [ ! -f \"server.jar\" ]; then\n    echo \">> PaperMC ($MINECRAFT_VERSION) のダウンロード中...\"\n    LATEST_BUILD=$(curl -s \"https://api.papermc.io/v2/projects/paper/versions/${MINECRAFT_VERSION}\" | jq '.builds[-1]' 2>/dev/null || true)\n    if [ \"$LATEST_BUILD\" != \"null\" ] && [ -n \"$LATEST_BUILD\" ]; then\n        DOWNLOAD_URL=\"https://api.papermc.io/v2/projects/paper/versions/${MINECRAFT_VERSION}/builds/${LATEST_BUILD}/downloads/paper-${MINECRAFT_VERSION}-${LATEST_BUILD}.jar\"\n        curl -sL -o server.jar \"$DOWNLOAD_URL\"\n    else\n        curl -sL -o server.jar \"https://api.purpurmc.org/v2/purpur/${MINECRAFT_VERSION}/latest/download\"\n    fi\nfi\n\n# 既存サーバープロセスの確認\nif pgrep -f \"server.jar\" > /dev/null 2>&1; then\n    echo \"✅ Minecraftサーバーは既にバックグラウンドで稼働中です。\"\nelse\n    echo \">> サーバープロセスをパイプ付きでバックグラウンド起動中...\"\n    PIPE_PATH=\"$SERVER_DIR/server.pipe\"\n    rm -f \"$PIPE_PATH\"\n    mkfifo \"$PIPE_PATH\"\n\n    # バックグラウンド起動 (nohup + tail -f FIFO パイプ)\n    nohup tail -f \"$PIPE_PATH\" | java -Xms2G -Xmx${MC_RAM_GB}G -XX:+UseG1GC -jar server.jar nogui > \"$SERVER_DIR/logs/latest.log\" 2>&1 &\n    MC_PID=$!\n    echo \"$MC_PID\" > /content/mc_server.pid\n\n    echo \"起動待機中 (約15秒)...\"\n    for i in {1..15}; do\n        if grep -q \"Done (\" \"$SERVER_DIR/logs/latest.log\" 2>/dev/null; then\n            break\n        fi\n        sleep 1\n        echo -n \".\"\n    done\n    echo \"\"\nfi\n\n# playit.gg トンネルの起動 (ポート25565の外部公開)\nif ! pgrep -f \"playit\" > /dev/null 2>&1; then\n    echo \">> playit.gg トンネルを起動中...\"\n    nohup playit --stdout > /content/playit.log 2>&1 &\n    sleep 3\nfi\n\nPLAYIT_URL=$(grep -oE \"([a-zA-Z0-9.-]+\\.ply\\.gg:[0-9]+|[a-zA-Z0-9.-]+\\.playit\\.plus)\" /content/playit.log 2>/dev/null | tail -n 1 || true)\nCLAIM_URL=$(grep -oE \"https://playit\\.gg/claim/[a-zA-Z0-9]+\" /content/playit.log 2>/dev/null | head -n 1 || true)\n\necho \"==================================================\"\necho \"✅ Minecraftサーバーはセルから独立してバックグラウンドで稼働しています。\"\necho \"ローカル接続: localhost:25565\"\nif [ -n \"$PLAYIT_URL\" ]; then\n    echo \"外部接続アドレス: $PLAYIT_URL\"\nelif [ -n \"$CLAIM_URL\" ]; then\n    echo \"playit トンネル割り当てリンク: $CLAIM_URL\"\nfi\necho \"==================================================\"\n", "scripts/record_screen.sh": "#!/bin/bash\n# ==============================================================================\n# record_screen.sh - Hardware-Accelerated (NVENC 60fps) Screen Recording\n# (Supports start, stop, status, toggle with PulseAudio sound)\n# ==============================================================================\nset -e\n\nACTION=\"${1:-status}\"\nPID_FILE=\"/content/recorder.pid\"\nRECORD_DIR=\"/content/recordings\"\nGDRIVE_REC=\"/content/drive/MyDrive/GamingPC/recordings\"\n\nmkdir -p \"$RECORD_DIR\"\n\nstart_recording() {\n    if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n        echo \"⚠️ 既に画面収録が実行中です (PID: $(cat \"$PID_FILE\"))\"\n        return 0\n    fi\n\n    TIMESTAMP=$(date +\"%Y%m%d_%H%M%S\")\n    OUT_FILE=\"$RECORD_DIR/Screen_${TIMESTAMP}.mp4\"\n    echo \">> 画面収録を開始します: $OUT_FILE\"\n\n    # PulseAudio モニターソース取得\n    AUDIO_SRC=$(pactl list short sources 2>/dev/null | grep -E \"VirtualSink|monitor\" | head -n 1 | awk '{print $2}' || true)\n    if [ -z \"$AUDIO_SRC\" ]; then\n        AUDIO_SRC=\"default\"\n    fi\n\n    # L4 GPU (NVENC) ハードウェアエンコード\n    # 音声と映像を同期して 1920x1080 60fps 収録\n    if ffmpeg -encoders 2>&1 | grep -q \"h264_nvenc\"; then\n        echo \">> NVIDIA NVENC ハードウェアアクセラレーションを使用します (L4 GPU)\"\n        nohup ffmpeg -y \\\n            -f x11grab -draw_mouse 1 -framerate 60 -video_size 1920x1080 -i :1.0 \\\n            -f pulse -i \"$AUDIO_SRC\" \\\n            -c:v h264_nvenc -preset p4 -tune ll -b:v 8M -maxrate 12M -bufsize 16M \\\n            -c:a aac -b:a 192k \\\n            -pix_fmt yuv420p \"$OUT_FILE\" > /content/ffmpeg_record.log 2>&1 &\n    else\n        echo \">> ソフトウェアエンコード (libx264) を使用します\"\n        nohup ffmpeg -y \\\n            -f x11grab -draw_mouse 1 -framerate 30 -video_size 1920x1080 -i :1.0 \\\n            -f pulse -i \"$AUDIO_SRC\" \\\n            -c:v libx264 -preset veryfast -crf 23 \\\n            -c:a aac -b:a 192k \\\n            -pix_fmt yuv420p \"$OUT_FILE\" > /content/ffmpeg_record.log 2>&1 &\n    fi\n\n    REC_PID=$!\n    echo \"$REC_PID\" > \"$PID_FILE\"\n    echo \"$OUT_FILE\" > /content/recorder_current_file.txt\n    sleep 2\n\n    if kill -0 \"$REC_PID\" 2>/dev/null; then\n        echo \"✅ 画面収録を開始しました (PID: $REC_PID)\"\n    else\n        echo \"⚠️ 収録開始に失敗しました。ログを確認してください:\"\n        tail -n 15 /content/ffmpeg_record.log 2>/dev/null || true\n    fi\n}\n\nstop_recording() {\n    if [ ! -f \"$PID_FILE\" ]; then\n        echo \"画面収録は実行されていません。\"\n        return 0\n    fi\n\n    REC_PID=$(cat \"$PID_FILE\")\n    if kill -0 \"$REC_PID\" 2>/dev/null; then\n        echo \">> 画面収録を終了・MP4ヘッダーを確定中 (PID: $REC_PID)...\"\n        kill -INT \"$REC_PID\" 2>/dev/null || true\n        # 終了待機\n        for i in {1..10}; do\n            if ! kill -0 \"$REC_PID\" 2>/dev/null; then\n                break\n            fi\n            sleep 1\n        done\n        echo \"✅ 画面収録を正常に終了しました。\"\n    fi\n    rm -f \"$PID_FILE\"\n\n    # Google Drive への同期\n    CURRENT_FILE=$(cat /content/recorder_current_file.txt 2>/dev/null || true)\n    if [ -n \"$CURRENT_FILE\" ] && [ -f \"$CURRENT_FILE\" ]; then\n        echo \"📁 録画ファイル: $CURRENT_FILE ($(ls -lh \"$CURRENT_FILE\" | awk '{print $5}'))\"\n        if [ -d \"/content/drive/MyDrive\" ]; then\n            mkdir -p \"$GDRIVE_REC\"\n            cp -f \"$CURRENT_FILE\" \"$GDRIVE_REC/\" 2>/dev/null || true\n            echo \"✅ Google Drive ($GDRIVE_REC) に同期しました。\"\n        fi\n    fi\n}\n\nstatus_recording() {\n    if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n        CURRENT_FILE=$(cat /content/recorder_current_file.txt 2>/dev/null || true)\n        echo \"🔴 画面収録中 (PID: $(cat \"$PID_FILE\"), ファイル: $CURRENT_FILE)\"\n    else\n        echo \"⚪ 画面収録は停止しています。\"\n    fi\n}\n\ncase \"$ACTION\" in\n    start)\n        start_recording\n        ;;\n    stop)\n        stop_recording\n        ;;\n    status)\n        status_recording\n        ;;\n    toggle)\n        if [ -f \"$PID_FILE\" ] && kill -0 \"$(cat \"$PID_FILE\")\" 2>/dev/null; then\n            stop_recording\n        else\n            start_recording\n        fi\n        ;;\n    *)\n        echo \"Usage: $0 {start|stop|status|toggle}\"\n        exit 1\n        ;;\nesac\n", "scripts/backup_restore.py": "#!/usr/bin/env python3\n\"\"\"\nbackup_restore.py - Google Drive Save Data & Profile Synchronization for Gaming PC\n(Syncs Minecraft worlds, Steam saves, Prism Launcher instances, Chrome profiles, and recordings)\n\"\"\"\n\nimport os\nimport sys\nimport shutil\nimport tarfile\nfrom datetime import datetime\nfrom pathlib import Path\n\nGDRIVE_BASE = Path(\"/content/drive/MyDrive/GamingPC\")\nGDRIVE_RECORDINGS = GDRIVE_BASE / \"recordings\"\n\n# バックアップ対象パスの定義 (存在する場合のみアーカイブ)\nTARGETS = [\n    {\"src\": Path(\"/content/minecraft_server/world\"), \"arcname\": \"minecraft_server/world\"},\n    {\"src\": Path(\"/content/minecraft_server/world_nether\"), \"arcname\": \"minecraft_server/world_nether\"},\n    {\"src\": Path(\"/content/minecraft_server/world_the_end\"), \"arcname\": \"minecraft_server/world_the_end\"},\n    {\"src\": Path(\"/content/minecraft_server/server.properties\"), \"arcname\": \"minecraft_server/server.properties\"},\n    {\"src\": Path(\"/content/minecraft_server/plugins\"), \"arcname\": \"minecraft_server/plugins\"},\n    {\"src\": Path(\"/content/chrome_profiles\"), \"arcname\": \"chrome_profiles\"},\n    {\"src\": Path.home() / \".local/share/PrismLauncher/instances\", \"arcname\": \"prismlauncher_instances\"},\n    {\"src\": Path.home() / \".steam/steam/userdata\", \"arcname\": \"steam_userdata\"},\n    {\"src\": Path.home() / \".config/discord\", \"arcname\": \"config_discord\"},\n]\n\ndef log(msg: str):\n    print(f\"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\ndef restore_from_storage():\n    \"\"\"Google Driveからセーブデータ・プロファイルを復元\"\"\"\n    if not GDRIVE_BASE.exists():\n        log(\"Google Driveのマウントまたはバックアップディレクトリが見つかりません。新規環境として開始します。\")\n        return False\n\n    latest_archive = GDRIVE_BASE / \"gaming_data_latest.tar.gz\"\n    if not latest_archive.exists():\n        # アーカイブ一覧から最新を検索\n        archives = sorted(GDRIVE_BASE.glob(\"gaming_data_*.tar.gz\"), key=os.path.getmtime, reverse=True)\n        if archives:\n            latest_archive = archives[0]\n\n    if latest_archive.exists():\n        log(f\"📦 Google Driveから最新バックアップを復元中: {latest_archive.name}\")\n        with tarfile.open(latest_archive, \"r:gz\") as tar:\n            for member in tar.getmembers():\n                dest_base = None\n                rel_path = None\n\n                if member.name.startswith(\"minecraft_server/\"):\n                    dest_base = Path(\"/content/minecraft_server\")\n                    rel_path = member.name[len(\"minecraft_server/\"):]\n                elif member.name.startswith(\"chrome_profiles\"):\n                    dest_base = Path(\"/content\")\n                    rel_path = member.name\n                elif member.name.startswith(\"prismlauncher_instances\"):\n                    dest_base = Path.home() / \".local/share/PrismLauncher\"\n                    rel_path = \"instances\" + member.name[len(\"prismlauncher_instances\"):]\n                elif member.name.startswith(\"steam_userdata\"):\n                    dest_base = Path.home() / \".steam/steam\"\n                    rel_path = \"userdata\" + member.name[len(\"steam_userdata\"):]\n                elif member.name.startswith(\"config_discord\"):\n                    dest_base = Path.home() / \".config\"\n                    rel_path = \"discord\" + member.name[len(\"config_discord\"):]\n\n                if dest_base and rel_path:\n                    target_file = dest_base / rel_path\n                    target_file.parent.mkdir(parents=True, exist_ok=True)\n                    if member.isdir():\n                        target_file.mkdir(parents=True, exist_ok=True)\n                    else:\n                        f = tar.extractfile(member)\n                        if f:\n                            with open(target_file, \"wb\") as out:\n                                shutil.copyfileobj(f, out)\n\n        log(\"✅ セーブデータ・プロファイルの復元が完了しました。\")\n        return True\n\n    log(\"復元対象のアーカイブがありませんでした。\")\n    return False\n\ndef backup_to_storage():\n    \"\"\"セーブデータ・プロファイル・録画動画をGoogle Driveへ完全同期\"\"\"\n    if not Path(\"/content/drive/MyDrive\").exists():\n        log(\"⚠️ Google Driveがマウントされていないため、Drive同期をスキップします。\")\n        return None\n\n    GDRIVE_BASE.mkdir(parents=True, exist_ok=True)\n    GDRIVE_RECORDINGS.mkdir(parents=True, exist_ok=True)\n\n    timestamp = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    archive_name = f\"gaming_data_{timestamp}.tar.gz\"\n    archive_path = GDRIVE_BASE / archive_name\n    latest_path = GDRIVE_BASE / \"gaming_data_latest.tar.gz\"\n\n    log(f\"📦 セーブデータと設定を圧縮バックアップ中: {archive_name}\")\n    has_items = False\n    with tarfile.open(archive_path, \"w:gz\") as tar:\n        for t in TARGETS:\n            src = t[\"src\"]\n            arcname = t[\"arcname\"]\n            if src.exists():\n                tar.add(src, arcname=arcname)\n                has_items = True\n\n    if has_items:\n        shutil.copy2(archive_path, latest_path)\n        log(\"✅ セーブデータアーカイブをGoogle Driveへ保存しました。\")\n    else:\n        log(\"保存対象のデータがまだありません。\")\n\n    # 画面収録動画 (MP4) の同期\n    local_rec = Path(\"/content/recordings\")\n    if local_rec.exists():\n        for mp4 in local_rec.glob(\"*.mp4\"):\n            dest_mp4 = GDRIVE_RECORDINGS / mp4.name\n            if not dest_mp4.exists() or dest_mp4.stat().st_size != mp4.stat().st_size:\n                log(f\"🎥 録画ファイルを同期中: {mp4.name} ({mp4.stat().st_size / (1024*1024):.1f} MB)\")\n                shutil.copy2(mp4, dest_mp4)\n        log(\"✅ 画面収録動画のGoogle Drive同期完了。\")\n\n    return latest_path\n\ndef backup_status():\n    \"\"\"現在のバックアップ状況を確認\"\"\"\n    if not GDRIVE_BASE.exists():\n        log(\"Google Drive同期ディレクトリが存在しません。\")\n        return\n\n    archives = list(GDRIVE_BASE.glob(\"gaming_data_*.tar.gz\"))\n    recordings = list(GDRIVE_RECORDINGS.glob(\"*.mp4\")) if GDRIVE_RECORDINGS.exists() else []\n\n    log(f\"Google Drive上のバックアップ数: {len(archives)} 個\")\n    for a in sorted(archives, key=os.path.getmtime, reverse=True)[:3]:\n        log(f\"  - {a.name} ({a.stat().st_size / (1024*1024):.2f} MB)\")\n\n    log(f\"Google Drive上の録画動画数: {len(recordings)} 本\")\n    for r in sorted(recordings, key=os.path.getmtime, reverse=True)[:3]:\n        log(f\"  - {r.name} ({r.stat().st_size / (1024*1024):.2f} MB)\")\n\nif __name__ == \"__main__\":\n    action = sys.argv[1] if len(sys.argv) > 1 else \"backup\"\n    if action == \"restore\":\n        restore_from_storage()\n    elif action == \"backup\":\n        backup_to_storage()\n    elif action == \"status\":\n        backup_status()\n    else:\n        print(f\"Unknown action: {action}\")\n", "scripts/shutdown.sh": "#!/bin/bash\n# ==============================================================================\n# shutdown.sh - Safe Shutdown & Complete Data Synchronization\n# (Stops recording, saves Minecraft, syncs to Google Drive, terminates daemons)\n# ==============================================================================\n\nREPO_DIR=\"$(cd \"$(dirname \"${BASH_SOURCE[0]}\")/..\" && pwd)\"\nSERVER_DIR=\"/content/minecraft_server\"\nPIPE_PATH=\"$SERVER_DIR/server.pipe\"\n\necho \"==================================================\"\necho \"  🛑 Gaming PC - 安全なシャットダウン処理開始\"\necho \"==================================================\"\n\n# 1. 画面収録プロセスの停止\necho \">> [1/5] 画面収録プロセスの終了とMP4の確定...\"\nbash \"$REPO_DIR/scripts/record_screen.sh\" stop 2>/dev/null || true\n\n# 2. 定期バックアッププロセスの終了\npkill -f \"backup_restore.py\" 2>/dev/null || true\n\n# 3. Minecraft サーバーのワールド保存と安全な停止\necho \">> [2/5] Minecraftサーバーのワールド保存 (save-all) & 停止...\"\nif [ -p \"$PIPE_PATH\" ] && pgrep -f \"server.jar\" > /dev/null 2>&1; then\n    echo \"say サーバーを安全にシャットダウンします...\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 1\n    echo \"save-all\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 2\n    echo \"stop\" > \"$PIPE_PATH\" 2>/dev/null || true\n    sleep 3\nfi\n\n# サーバープロセス終了待機\nfor i in {1..15}; do\n    if ! pgrep -f \"server.jar\" > /dev/null 2>&1; then\n        break\n    fi\n    sleep 1\n    echo -n \".\"\ndone\necho \"\"\npkill -TERM -f \"server.jar\" 2>/dev/null || true\nrm -f \"$PIPE_PATH\" /content/mc_server.pid\n\n# 4. セーブデータ & 録画動画のGoogle Drive同期\necho \">> [3/5] セーブデータ・プロファイル・録画動画の最終Google Drive同期...\"\npython3 \"$REPO_DIR/scripts/backup_restore.py\" backup || true\n\n# 5. トンネルおよびデスクトッププロセスの終了\necho \">> [4/5] トンネル & リモートデスクトッププロセスの停止...\"\npkill -f \"playit\" 2>/dev/null || true\npkill -f \"cloudflared tunnel\" 2>/dev/null || true\npkill -f \"websockify\" 2>/dev/null || true\npkill -f \"x11vnc\" 2>/dev/null || true\npkill -f \"xfce4\" 2>/dev/null || true\npkill -f \"Xvfb :1\" 2>/dev/null || true\n\necho \">> [5/5] シャットダウン完了確認...\"\necho \"\"\necho \"==================================================\"\necho \"🎉 すべてのデータ保存とシャットダウンが完了しました！\"\necho \"💾 保存先: Google Drive > GamingPC/\"\necho \"お疲れ様でした。\"\necho \"==================================================\"\n"}

for rel_path, content in EMBEDDED_FILES.items():
    full_path = os.path.join(REPO_DIR, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, "w", encoding="utf-8") as f:
        f.write(content)
    if full_path.endswith(".sh") or full_path.endswith(".py"):
        os.chmod(full_path, 0o755)

print("✅ システムスクリプトの展開完了")

# 3. 仮想環境セットアップ
print("\n>> [1/5] システム依存関係 & アプリケーションのセットアップ中 (約1分)...\n")
setup_cmd = f"bash '{SCRIPTS_DIR}/setup_environment.sh'"
p = subprocess.run(setup_cmd, shell=True)
if p.returncode != 0:
    print("⚠️ setup_environment.sh で一部警告が出ましたが処理を継続します。")

# 4. Google Driveからのセーブデータ復元
print("\n>> [2/5] Google Driveから以前のセーブデータ & 設定を復元中...")
subprocess.run(f"python3 '{SCRIPTS_DIR}/backup_restore.py' restore", shell=True)

# 5. デスクトップ環境 & リモート接続の起動
print("\n>> [3/5] デスクトップ環境 (Xfce4) & リモート接続をバックグラウンド起動中...")
crd_arg = f"'{CRD_AUTH_CODE}'" if CRD_AUTH_CODE else "''"
start_desktop_cmd = f"bash '{SCRIPTS_DIR}/start_desktop.sh' {crd_arg} '{CRD_PIN}'"
subprocess.run(start_desktop_cmd, shell=True)

# 6. Minecraftサーバーのバックグラウンド起動 (独立稼働)
if START_MINECRAFT_SERVER:
    print("\n>> [4/5] 独立Minecraftサーバー (PaperMC) をバックグラウンド起動中...")
    start_mc_cmd = f"bash '{SCRIPTS_DIR}/start_mc_server.sh' {MC_RAM_GB}"
    subprocess.run(start_mc_cmd, shell=True)
else:
    print("\n>> [4/5] Minecraftサーバーの自動起動はスキップされました。")

# 7. 定期自動バックアップデーモンの起動 (15分間隔)
print("\n>> [5/5] バックグラウンド定期自動バックアップ監視を起動中...")
subprocess.run(f"nohup bash -c 'while true; do sleep 900; python3 \"{SCRIPTS_DIR}/backup_restore.py\" backup > /content/autobackup.log 2>&1; done' > /dev/null 2>&1 &", shell=True)

# 8. 接続URLとダッシュボードの表示
time.sleep(3)
novnc_url = ""
if os.path.exists("/content/cloudflared.log"):
    with open("/content/cloudflared.log", "r") as f:
        log_text = f.read()
        import re
        m = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_text)
        if m:
            novnc_url = m[-1] + "/vnc.html?autoconnect=true&resize=scale"

mc_address = "localhost:25565"
if os.path.exists("/content/playit.log"):
    with open("/content/playit.log", "r") as f:
        log_text = f.read()
        import re
        m = re.findall(r"([a-zA-Z0-9.-]+\.ply\\.gg:[0-9]+|[a-zA-Z0-9.-]+\.playit\\.plus)", log_text)
        if m:
            mc_address = m[-1]

html_card = f"""
<div style="font-family: sans-serif; background: linear-gradient(135deg, #1e1e2f, #161621); color: #fff; padding: 24px; border-radius: 12px; border: 1px solid #3d3d5c; box-shadow: 0 8px 24px rgba(0,0,0,0.5);">
    <h2 style="margin-top:0; color: #4ade80; display:flex; align-items:center; gap: 8px;">
        🎮 Colab Gaming PC - 起動完了！
    </h2>
    <p style="color: #cbd5e1; font-size: 14px; margin-bottom: 20px;">
        すべてのゲーム・デスクトップ・Minecraftサーバーがセルから独立してバックグラウンド稼働中です。<br>
        <strong>このセルの実行は完了しました。他のセルやターミナルを自由にご利用いただけます。</strong>
    </p>

    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap: 16px; margin-bottom: 20px;">
        <div style="background: rgba(255,255,255,0.06); padding: 16px; border-radius: 8px; border-left: 4px solid #38bdf8;">
            <h3 style="margin: 0 0 8px 0; color: #38bdf8; font-size: 16px;">🌐 Webブラウザで直接接続 (noVNC)</h3>
            <p style="margin: 0 0 12px 0; font-size: 13px; color: #94a3b8;">専用ソフト不要・ワンクリックでデスクトップを操作:</p>
            {f'<a href="{novnc_url}" target="_blank" style="display: inline-block; background: #38bdf8; color: #0f172a; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; font-size: 14px;">🖥️ noVNC デスクトップを開く</a>' if novnc_url else '<span style="color:#f87171;">トンネルURLを取得中... ステップ2で再確認してください</span>'}
        </div>

        <div style="background: rgba(255,255,255,0.06); padding: 16px; border-radius: 8px; border-left: 4px solid #a855f7;">
            <h3 style="margin: 0 0 8px 0; color: #c084fc; font-size: 16px;">🖥️ Chrome Remote Desktop</h3>
            <p style="margin: 0 0 8px 0; font-size: 13px; color: #94a3b8;">高画質・ゲームプレイ推奨 (コード設定時):</p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="display: inline-block; background: #a855f7; color: #fff; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; font-size: 14px;">🚀 Google Remote Desktop を開く</a>
            <div style="font-size: 12px; color: #94a3b8; margin-top: 6px;">設定PIN: <code>{CRD_PIN}</code> (※後から設定する場合はステップ2を実行)</div>
        </div>

        <div style="background: rgba(255,255,255,0.06); padding: 16px; border-radius: 8px; border-left: 4px solid #22c55e;">
            <h3 style="margin: 0 0 8px 0; color: #4ade80; font-size: 16px;">⛏️ Minecraft サーバー接続先</h3>
            <p style="margin: 0 0 8px 0; font-size: 13px; color: #94a3b8;">手元のMinecraft Java版クライアントから接続:</p>
            <div style="background: rgba(0,0,0,0.4); padding: 8px 12px; border-radius: 6px; font-family: monospace; font-size: 15px; color: #86efac; border: 1px dashed #22c55e;">
                {mc_address}
            </div>
        </div>
    </div>

    <div style="background: rgba(0,0,0,0.3); padding: 14px; border-radius: 8px; font-size: 13px; line-height: 1.6; color: #cbd5e1;">
        <strong>📌 デスクトップ内の主要機能:</strong><br>
        • <strong>Google Chrome</strong>: Profile 1 / 2 / 3 の個別ショートカット（マルチタブ展開可能）<br>
        • <strong>Steam & CurseForge</strong>: デスクトップから即座に起動・ゲーム/MOD管理<br>
        • <strong>Minecraft (Prism Launcher)</strong>: Modrinth/CurseForgeのMODパックをワンクリック導入<br>
        • <strong>Xbox Cloud Gaming / Roblox / ChillwithYou</strong>: 専用ショートカット完備<br>
        • <strong>🎥 画面収録</strong>: デスクトップ上のショートカット、またはステップ5からワンクリックでNVENC 60fps録画開始/停止<br>
        • <strong>💾 データ保存</strong>: プレイ終了時は必ず「ステップ 6」を実行してGoogle Driveへ安全保存してください。
    </div>
</div>
"""
display(HTML(html_card))


---
## 🖥️ ステップ 2: 【追加接続】Chrome Remote Desktop のセットアップ（後から追加する場合）
環境起動後にChrome Remote Desktopを追加で有効化したい場合や、コードを後から入力したい場合はこのセルを実行します。

1. お手元のブラウザで [Google Remote Desktop Headless (SSH)](https://remotedesktop.google.com/headless) を開きます。
2. 「開始」→「次へ」→「承認」をクリックし、**Debian Linux** 用に表示されたコードまたはコマンド全体をコピーします。
3. 下の `CRD_AUTH_CODE` に貼り付けて実行します。
4. 実行後、手元のChromeリモートデスクトップアプリまたは [remotedesktop.google.com/access](https://remotedesktop.google.com/access) から接続します。

In [ ]:
#@title 🖥️ ステップ 2: Chrome Remote Desktop の構成・起動
import os
import subprocess

#@markdown ### 認証コマンド または 認証コード (remotedesktop.google.com/headless からコピー)
CRD_AUTH_CODE = "" #@param {type:"string"}
CRD_PIN = "123456" #@param {type:"string"}

if not CRD_AUTH_CODE.strip():
    print("⚠️ CRD_AUTH_CODE が入力されていません。https://remotedesktop.google.com/headless から取得したコードを入力してください。")
else:
    SCRIPTS_DIR = "/content/ColabTool/GamingPC/scripts"
    crd_arg = f"'{CRD_AUTH_CODE.strip()}'"
    print(">> Chrome Remote Desktop ホストを構成中...")
    subprocess.run(f"bash '{SCRIPTS_DIR}/start_desktop.sh' {crd_arg} '{CRD_PIN}'", shell=True)
    print("\n==================================================")
    print("✅ Chrome Remote Desktop の登録が完了しました！")
    print(f"👉 https://remotedesktop.google.com/access を開いて接続してください (PIN: {CRD_PIN})")
    print("==================================================")


---
## 📊 ステップ 3: 稼働状況チェック＆診断（任意実行）
システムリソース（GPU VRAM使用量、CPU、メモリ、ディスク残量）や、各種バックグラウンドプロセスの稼働状態を確認できます。  
必要に応じて、任意のBashコマンドを実行することも可能です。

In [ ]:
#@title 📊 ステップ 3: 稼働状況チェック＆診断（任意実行）
import os
import subprocess

#@markdown ### 任意コマンドの実行（空欄の場合はステータス診断のみ表示）
BASH_COMMAND = "" #@param {type:"string"}

if BASH_COMMAND.strip():
    print(f">> コマンド実行: {BASH_COMMAND}")
    subprocess.run(BASH_COMMAND, shell=True)
    print("--------------------------------------------------")

print("=== [1] NVIDIA L4 GPU ステータス ===")
subprocess.run("nvidia-smi --query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu --format=csv", shell=True)

print("\n=== [2] システムリソース (CPU / RAM / Disk) ===")
subprocess.run("free -h && echo '' && df -h /content", shell=True)

print("\n=== [3] 主要バックグラウンドプロセスの稼働状況 ===")
services = {
    "Xvfb (仮想ディスプレイ)": "Xvfb :1",
    "Xfce4 (デスクトップ)": "xfce4-session",
    "x11vnc (VNCサーバー)": "x11vnc",
    "noVNC (Webソケット)": "websockify",
    "CRD (リモートホスト)": "chrome-remote-desktop",
    "Cloudflared (トンネル)": "cloudflared tunnel",
    "Minecraft サーバー": "server.jar",
    "playit.gg (トンネル)": "playit",
    "画面収録 (NVENC)": "record_screen.sh",
}

for name, pattern in services.items():
    res = subprocess.run(f"pgrep -f '{pattern}'", shell=True, capture_output=True, text=True)
    if res.returncode == 0:
        pids = ",".join(res.stdout.strip().split("\n"))
        print(f"  ✅ {name:<22}: 稼働中 (PID: {pids})")
    else:
        print(f"  ⚪ {name:<22}: 停止中")

print("\n=== [4] 最新接続情報 ===")
import re
try:
    if os.path.exists("/content/cloudflared.log"):
        with open("/content/cloudflared.log") as f:
            m = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
            if m:
                print(f"  🌐 noVNC URL: {m[-1]}/vnc.html?autoconnect=true&resize=scale")
            else:
                print("  🌐 noVNC URL: トンネル接続確立中...")
    if os.path.exists("/content/playit.log"):
        with open("/content/playit.log") as f:
            p_text = f.read()
            m = re.findall(r"([a-zA-Z0-9.-]+\.ply\.gg:[0-9]+|[a-zA-Z0-9.-]+\.playit\.plus)", p_text)
            claim = re.findall(r"https://playit\.gg/claim/[a-zA-Z0-9]+", p_text)
            if m:
                print(f"  ⛏️ Minecraft 接続アドレス: {m[-1]}")
            elif claim:
                print(f"  ⛏️ playit.gg トンネル登録URL: {claim[0]}")
            else:
                print("  ⛏️ Minecraft 接続: localhost:25565")
except Exception as e:
    print("接続情報取得エラー:", e)


---
## ⛏️ ステップ 4: Minecraftサーバー コマンド送信＆ログ確認（任意実行）
バックグラウンドで稼働しているMinecraftサーバーに対して、セルの実行を妨げることなくコマンド（`op`, `say`, `whitelist`, `save-all` など）を直接送信したり、直近のサーバーログを確認できます。

In [ ]:
#@title ⛏️ ステップ 4: Minecraftサーバー コマンド送信＆ログ確認（任意実行）
import os
import subprocess
import time

#@markdown ### サーバーコマンド送信 (例: `say こんにちは`, `list`, `save-all`, `op プレイヤー名`)
MC_COMMAND = "list" #@param {type:"string"}
VIEW_LOG_LINES = 30 #@param {type:"integer"}

PIPE_PATH = "/content/minecraft_server/server.pipe"
LOG_PATH = "/content/minecraft_server/logs/latest.log"

if os.path.exists(PIPE_PATH):
    if MC_COMMAND.strip():
        print(f">> コマンド送信: {MC_COMMAND}")
        with open(PIPE_PATH, "w") as pipe:
            pipe.write(MC_COMMAND.strip() + "\n")
        time.sleep(1)
else:
    print("⚠️ Minecraftサーバーのパイプが見つかりません。サーバーが稼働しているか確認してください。")

print(f"\n=== 直近 {VIEW_LOG_LINES} 行のサーバーログ ===")
if os.path.exists(LOG_PATH):
    subprocess.run(f"tail -n {VIEW_LOG_LINES} '{LOG_PATH}'", shell=True)
else:
    print("ログファイルがまだ生成されていません。")


---
## 🎥 ステップ 5: 画面収録 (NVENC 60fps) コントロール（任意実行）
NVIDIA L4 GPUのハードウェアエンコーダ（NVENC）を利用し、負荷なくゲーム画面と音声を高画質（1080p 60fps）でMP4録画できます。  
デスクトップ上のショートカット「🎥 画面収録」からも手動で切り替え可能です。

In [ ]:
#@title 🎥 ステップ 5: 画面収録 (NVENC 60fps) コントロール（任意実行）
import subprocess

#@markdown ### 録画アクション
RECORD_ACTION = "status" #@param ["status", "start", "stop", "toggle"]

SCRIPTS_DIR = "/content/ColabTool/GamingPC/scripts"
subprocess.run(f"bash '{SCRIPTS_DIR}/record_screen.sh' {RECORD_ACTION}", shell=True)

print("\n📁 現在のローカル録画ファイル一覧 (/content/recordings/):")
subprocess.run("ls -lh /content/recordings/*.mp4 2>/dev/null || echo '（録画ファイルはまだありません）'", shell=True)


---
## 🛑 ステップ 6: 【ワンクリック安全停止】セーブ同期＆シャットダウン
ゲームプレイ終了時は、必ずこのセルを実行してください。  
以下の処理が全自動で行われ、データ消失を防ぎます：
1. 実行中の画面収録を正常に終了・MP4ヘッダーを確定
2. Minecraftサーバーのワールドデータを確実に保存 (`save-all`) して停止
3. ワールド、Steamセーブ、ブラウザプロファイル、MOD構成、録画動画を **Google Drive (`/content/drive/MyDrive/GamingPC/`) へ完全同期**
4. すべてのバックグラウンドプロセスを安全に終了

In [ ]:
#@title 🛑 ステップ 6: 【ワンクリック安全停止】セーブ同期＆シャットダウン
import os
import subprocess

#@markdown ### 停止後のランタイム切断設定
DISCONNECT_RUNTIME = False #@param {type:"boolean"}

print("==================================================")
print("  🛑 Gaming PC - 安全停止 & Google Drive完全同期")
print("==================================================")

SCRIPTS_DIR = "/content/ColabTool/GamingPC/scripts"
subprocess.run(f"bash '{SCRIPTS_DIR}/shutdown.sh'", shell=True)

# Google Drive上のバックアップ状況
print("\n📁 Google Drive上の最新バックアップ状況:")
subprocess.run(f"python3 '{SCRIPTS_DIR}/backup_restore.py' status", shell=True)

print("\n==================================================")
print("✅ すべてのセーブデータが安全にGoogle Driveへ同期されました。")
print("==================================================")

if DISCONNECT_RUNTIME:
    try:
        from google.colab import runtime
        print("🔌 ランタイムを切断して割り当てを解除します...")
        runtime.unassign()
    except Exception as e:
        print("ランタイム自動切断スキップ:", e)
